In [3]:
import datetime

import colormaps
import matplotlib.pyplot as plt
import os
import numpy as np
import polars as pl
import xarray as xr
from jetutils.anyspell import get_persistent_jet_spells, mask_from_spells_pl, subset_around_onset, get_persistent_spell_times_from_som, get_spells, extend_spells, gb_index, make_daily
from jetutils.clustering import Experiment, RAW_REALSPACE, labels_to_centers
from jetutils.data import *
from jetutils.geospatial import *
from jetutils.definitions import *
from jetutils.jet_finding import *
from jetutils.plots import *
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from tqdm import tqdm
import polars.selectors as cs

os.environ["RUST_BACKTRACE"] = "1"

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

IPython could not be loaded!


# rest

In [22]:
# ds_cesm = xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/historical/ds.zarr", engine="zarr")
# ds_cesm = standardize(ds_cesm)
# ds_cesm["theta"] = (ds_cesm["t"] * (1000 / ds_cesm["lev"]) ** KAPPA).astype(np.float32)
# ds_cesm = ds_cesm.drop_vars("t")
# dh = DataHandler.from_basepath_and_da("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/historical/results", ds_cesm)
# exp = JetFindingExperiment(dh)
# all_jets_one_df = exp.find_jets()
# props_as_df = exp.props_as_df()

In [ ]:
ds_future = xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/ssp370/ds.zarr", engine="zarr")
ds_future = standardize(ds_future)
# ds_future = extract(ds_future, period=(2060, 2071), members=[1, 2, 3, 4])
ds_future["theta"] = (ds_future["t"] * (1000 / ds_future["lev"]) ** KAPPA).astype(np.float32)
ds_future = ds_future.drop_vars("t")
ds_future = ds_future.load()
jets = find_all_jets(ds_future, n_coarsen=1, hole_size=3)

In [ ]:
both_jets = {}
both_paths = {}
for run in ["historical", "ssp370"]:
    ds = xr.open_dataset(f"/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/{run}/ds.zarr", engine="zarr")
    ds = standardize(ds)
    ds = extract(ds, (1970, 1981), members=[1, 2, 3])
    ds["theta"] = (ds["t"] * (1000 / ds["lev"]) ** KAPPA).astype(np.float32)
    ds = ds.drop_vars("t")
    dh = DataHandler.from_basepath_and_da(f"/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/{run}/results", ds)
    exp = JetFindingExperiment(dh)
    all_jets_one_df = exp.find_jets(force=False, n_coarsen=1, smooth_s=5, alignment_thresh=0.55, base_s_thresh=0.55, int_thresh_factor=0.2, hole_size=5)
    if "is_polar" not in all_jets_one_df.columns:
        all_jets_one_df = exp.categorize_jets(None, ["s", "theta"], force=False, n_init=15, init_params="k-means++", mode="month").cast({"time": pl.Datetime("ms")})
    exp.props_as_df()
    
    phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["member", "time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["member", "time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["member", "time", "jet ID"]) > 1.3e8)))

    phat_jets_catd = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["member", "time", "jet ID"]) > 0.5).cast(pl.UInt32())})
    
    cross_catd_ofile = exp.path.joinpath("cross_catd.parquet")
    if cross_catd_ofile.is_file():
        cross_catd = pl.read_parquet(cross_catd_ofile)
    else:
        cross_catd = track_jets(phat_jets_catd)
        cross_catd = pers_from_cross_catd(cross_catd)
        cross_catd.write_parquet(cross_catd_ofile)


    # phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["time", "jet ID"]) > 1.e8)))
    # both_paths[run] = exp.path
    # both_jets[run] = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5).cast(pl.UInt32())})

100%|██████████| 36/36 [03:11<00:00,  5.33s/it]


In [7]:
cross_all = track_jets(all_jets_one_df)

100%|██████████| 3/3 [00:18<00:00,  6.01s/it]


In [8]:
cross_all

member,time,jet ID,s,theta,is_polar,time_right,jet ID_right,s_right,theta_right,is_polar_right,dist,lon_overlap,ds,dtheta,dis_polar
str,datetime[ms],u32,f32,f32,f64,datetime[ms],u32,f32,f32,f64,f64,f64,f32,f32,f64
"""r10i1251p1f1""",1970-01-01 12:00:00,0,45.865166,340.216949,0.210645,1970-01-02 12:00:00,0,44.856083,343.772278,0.208076,801753.346367,0.857143,1.009083,3.555328,0.002569
"""r10i1251p1f1""",1970-01-01 12:00:00,0,45.865166,340.216949,0.210645,1970-01-02 12:00:00,1,58.952141,348.66568,0.335226,5.2877e6,0.0,13.086975,8.44873,0.124581
"""r10i1251p1f1""",1970-01-01 12:00:00,0,45.865166,340.216949,0.210645,1970-01-02 12:00:00,2,45.211224,323.111938,0.760147,5.2486e6,0.346939,0.653942,17.105011,0.549502
"""r10i1251p1f1""",1970-01-01 12:00:00,0,45.865166,340.216949,0.210645,1970-01-02 12:00:00,3,42.18137,317.425812,0.874696,8.0160e6,0.0,3.683796,22.791138,0.664051
"""r10i1251p1f1""",1970-01-01 12:00:00,1,65.291718,345.148895,0.297504,1970-01-02 12:00:00,0,44.856083,343.772278,0.208076,4.8702e6,0.0,20.435635,1.376617,0.089429
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""r10i1231p1f1""",1981-12-30 12:00:00,1,62.483967,332.305206,0.350896,1981-12-31 12:00:00,1,56.318436,339.215057,0.186665,7.9714e6,0.964286,6.165531,6.909851,0.164231
"""r10i1231p1f1""",1981-12-30 12:00:00,1,62.483967,332.305206,0.350896,1981-12-31 12:00:00,2,56.816936,326.339264,0.410243,3.3805e6,0.84,5.66703,5.965942,0.059347
"""r10i1231p1f1""",1981-12-30 12:00:00,2,32.081501,305.110657,0.701544,1981-12-31 12:00:00,0,50.16671,345.609558,0.285655,1.3558e7,1.0,18.085209,40.498901,0.415889


In [9]:
pers_from_cross_catd(cross_all)

member,time,jet ID,s,theta,is_polar,time_right,jet ID_right,s_right,theta_right,is_polar_right,dist,lon_overlap,ds,dtheta,dis_polar,spell_of,pers
str,datetime[ms],u32,f32,f32,f64,datetime[ms],u32,f32,f32,f64,f64,f64,f32,f32,f64,str,f64
"""r10i1251p1f1""",1970-01-01 12:00:00,0,45.865166,340.216949,0.210645,1970-01-02 12:00:00,0,44.856083,343.772278,0.208076,801753.346367,0.857143,1.009083,3.555328,0.002569,"""STJ""",0.0
"""r10i1251p1f1""",1970-01-01 12:00:00,1,65.291718,345.148895,0.297504,1970-01-02 12:00:00,1,58.952141,348.66568,0.335226,177531.917993,0.888889,6.339577,3.516785,0.037722,"""EDJ""",0.0
"""r10i1251p1f1""",1970-01-01 12:00:00,2,46.168133,322.384705,0.778628,1970-01-02 12:00:00,2,45.211224,323.111938,0.760147,1.1577e6,0.897959,0.956909,0.727234,0.018481,"""EDJ""",0.0
"""r10i1251p1f1""",1970-01-02 12:00:00,0,44.856083,343.772278,0.208076,1970-01-03 12:00:00,0,47.21772,345.200317,0.224476,5.0143e6,0.0,2.361637,1.42804,0.0164,"""STJ""",0.0
"""r10i1251p1f1""",1970-01-02 12:00:00,1,58.952141,348.66568,0.335226,1970-01-03 12:00:00,1,51.11578,348.162933,0.282382,3.6179e6,0.490196,7.836361,0.502747,0.052845,"""EDJ""",0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""r10i1231p1f1""",1981-12-29 12:00:00,1,66.517624,334.161255,0.363636,1981-12-30 12:00:00,1,62.483967,332.305206,0.350896,3.5339e6,0.768293,4.033657,1.856049,0.01274,"""EDJ""",0.0
"""r10i1231p1f1""",1981-12-29 12:00:00,2,35.567005,303.142517,0.706815,1981-12-30 12:00:00,2,32.081501,305.110657,0.701544,4.4526e6,0.833333,3.485504,1.96814,0.005271,"""EDJ""",0.0
"""r10i1231p1f1""",1981-12-30 12:00:00,0,56.362705,343.262146,0.287249,1981-12-31 12:00:00,0,50.16671,345.609558,0.285655,490649.616788,1.0,6.195995,2.347412,0.001594,"""STJ""",0.0
